# 05 - Batch Inference

This notebook runs batch predictions using `model_version.run()` -- scalable inference executed directly on Snowflake warehouses with zero data movement.

**Key Concepts**:
- `model_version.run()` executes predictions on the warehouse
- Data never leaves Snowflake
- Vectorized execution for high throughput
- Results saved directly to tables

**Prerequisites**: Run `02_model_training_xgboost.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.ml.registry import Registry

session = get_active_session()
print(f"Connected: {session.get_current_user()}")

In [ ]:
# Load model from registry (using default version)
reg = Registry(
    session=session,
    database_name="MLOPS_DEMO_DB",
    schema_name="PIPELINES"
)

model = reg.get_model("CHURN_PREDICTION_MODEL")
mv = model.default
print(f"Loaded: {mv.model_name} version {mv.version_name}")

In [ ]:
# Prepare inference data
inference_df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS").select(
    "CUSTOMER_ID",
    "CREDIT_SCORE", "AGE", "TENURE", "BALANCE",
    "NUM_OF_PRODUCTS", "HAS_CR_CARD", "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"
)
print(f"Inference data: {inference_df.count()} rows")
inference_df.show(5)

In [ ]:
# Run batch predictions
# This executes on the warehouse -- data never leaves Snowflake
predictions = mv.run(
    inference_df,
    function_name="predict"
)
print("Predictions:")
predictions.show(10)

In [ ]:
# Run probability predictions
probabilities = mv.run(
    inference_df,
    function_name="predict_proba"
)
print("Prediction Probabilities:")
probabilities.show(10)

In [ ]:
# Add metadata columns and save to OUTPUT schema
results = predictions.with_column(
    "PREDICTION_TIMESTAMP", F.current_timestamp()
).with_column(
    "MODEL_VERSION", F.lit(mv.version_name)
)

results.write.save_as_table(
    "MLOPS_DEMO_DB.OUTPUT.CHURN_PREDICTIONS",
    mode="overwrite"
)
print("Predictions saved to MLOPS_DEMO_DB.OUTPUT.CHURN_PREDICTIONS")

In [ ]:
# Verify saved predictions
saved = session.table("MLOPS_DEMO_DB.OUTPUT.CHURN_PREDICTIONS")
print(f"Saved {saved.count()} predictions")
saved.show(5)

In [ ]:
# Prediction distribution
session.sql("""
SELECT 
    output_feature_0 AS PREDICTED_CHURN,
    COUNT(*) AS COUNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS PERCENTAGE
FROM MLOPS_DEMO_DB.OUTPUT.CHURN_PREDICTIONS
GROUP BY output_feature_0
ORDER BY output_feature_0
""").show()

## What to Show in SnowSight

Navigate to **Data > Databases > MLOPS_DEMO_DB > OUTPUT > CHURN_PREDICTIONS** to see:
- Prediction results with timestamps and model version
- Data preview showing predictions alongside customer IDs

**Key message**: Batch inference runs entirely within Snowflake -- no data export, no external compute, no latency from data movement.

---

**Next**: `06_model_monitoring.ipynb` - Set up drift and performance monitoring